In [ ]:
import sys, os
sys.path.insert(0, '/content/td-mpc_o2')
sys.path.insert(0, '/content/td-mpc_o2/tdmpc/src')
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
import torch
import numpy as np
import time
from pathlib import Path
from omegaconf import OmegaConf
from cfg import parse_cfg
from env import make_env
from algorithm.helper import Episode, ReplayBuffer, linear_schedule
from o2.tdmpc_o2 import TDMPC_O2
from o2.training_utils import set_seed, update_tdmpc, update_decoder

In [ ]:
O2_DEFAULTS = {
    'latent_action_dim': 128, 'decoder_init': True, 'use_latent_state': True,
    'dcem_batch_size': 64, 'decoder_updates': 50, 'told_updates': 500,
    'decoder_start_steps': 5000, 'exp_name': 'o2_ddpg',
    'latent_num_samples': 32, 'latent_num_elites': 8, 'dcem_sampling_n': None,
}

CFG_PATH = Path('/content/td-mpc_o2/tdmpc/cfgs')
sys.argv = ['train', 'task=walker-walk', 'seed=1']
cfg = parse_cfg(CFG_PATH)
cfg = OmegaConf.merge(OmegaConf.create(O2_DEFAULTS), cfg)

# Override anything here:
# cfg.lr = 3e-4
# cfg.decoder_start_steps = 2000

print(OmegaConf.to_yaml(cfg))

In [ ]:
set_seed(cfg.seed)
env    = make_env(cfg)
agent  = TDMPC_O2(cfg)
buffer = ReplayBuffer(cfg)
step, episode_idx = 0, 0
start_time = time.time()
print('Ready')

In [ ]:
if cfg.get('load_model', None):
    d = torch.load(cfg.load_model)
    state_dict = d['model'] if 'model' in d else d
    agent.model.load_state_dict(state_dict, strict=False)
    if 'model_target' in d:
        agent.model_target.load_state_dict(d['model_target'], strict=False)
    print(f'Loaded model from {cfg.load_model}')

if cfg.get('load_buffer', None):
    buffer.__dict__.update(torch.load(cfg.load_buffer, weights_only=False))
    print(f'Loaded buffer from {cfg.load_buffer}')

In [ ]:
num_episodes = 10
W = 38

for _ in range(num_episodes):
    phase = 'o2' if step >= cfg.decoder_start_steps else 'tdmpc'

    obs = env.reset()
    episode = Episode(cfg, obs)
    t_ep = time.time()
    while not episode.done:
        if step < cfg.seed_steps:
            action = torch.tensor(env.action_space.sample(), dtype=torch.float32, device=agent.device)
        elif phase == 'tdmpc':
            action = agent.plan(obs, step=step, t0=episode.first)
        else:
            action, *_ = agent.CEM_in_latent(obs, step=step, t0=episode.first, sample_final_action=True)
        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    buffer += episode
    ep_time = time.time() - t_ep

    train_metrics, update_time = {}, 0.0
    if step >= cfg.seed_steps:
        t = time.time()
        train_metrics = update_tdmpc(agent, buffer, step)
        update_time = time.time() - t

    dec_metrics = {}
    decoder_time = 0.0
    if phase == 'o2':
        t = time.time()
        dec_metrics = update_decoder(agent, buffer, cfg, step)
        decoder_time = time.time() - t
        for iteration, norm in sorted(dec_metrics['grad_tracker']):
            print(f'  DCEM iter {iteration} grad norm: {norm:.6f}')

    step += cfg.episode_length
    episode_idx += 1
    env_step = int(step * cfg.action_repeat)

    print('─' * W)
    print(f'  Episode {episode_idx}   step {env_step:,}   [{phase}]')
    print('─' * W)
    print(f'  {"Reward":<16}: {episode.cumulative_reward:>8.1f}')
    print(f'  {"Horizon":<16}: {int(linear_schedule(cfg.horizon_schedule, step)):>8}')
    print(f'  {"Std":<16}: {linear_schedule(cfg.std_schedule, step):>8.3f}')
    print(f'  {"Ep time":<16}: {ep_time:>7.1f}s')
    print(f'  {"Update time":<16}: {update_time:>7.1f}s')
    if dec_metrics:
        print(f'  {"Decoder time":<16}: {decoder_time:>7.1f}s')
        print(f'  {"Decoder loss":<16}: {dec_metrics.get("decoder_loss", 0):>8.4f}')
        print(f'  {"Decoder grad":<16}: {dec_metrics.get("decoder_grad_norm", 0):>8.4f}')
        print(f'  {"Saturation":<16}: {dec_metrics.get("saturation", 0):>8.4f}')
    print(f'  {"Total time":<16}: {time.time() - start_time:>7.0f}s')
    if train_metrics:
        print(f'  {"total_loss":<16}: {train_metrics.get("total_loss", 0):>8.3f}')
        print(f'  {"reward_loss":<16}: {train_metrics.get("reward_loss", 0):>8.3f}')
        print(f'  {"value_loss":<16}: {train_metrics.get("value_loss", 0):>8.3f}')
        print(f'  {"pi_loss":<16}: {train_metrics.get("pi_loss", 0):>8.3f}')